In [ ]:
import pandas as pd
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import pickle

# 1. Download multi-stock data (better model)
stocks = ["AAPL", "MSFT", "TSLA"]
df = yf.download(stocks, period="5y", group_by='ticker')

# 2. Combine data
frames = []
for stock in stocks:
    temp = df[stock].copy()
    temp['Stock'] = stock
    frames.append(temp)

df = pd.concat(frames)

# 3. Feature Engineering
df['MA10'] = df['Close'].rolling(10).mean()
df['MA50'] = df['Close'].rolling(50).mean()
df['Return'] = df['Close'].pct_change()

# 4. Target
df['Target'] = df['Close'].shift(-1)

df = df.dropna()

# 5. Features & Label
X = df[['Open','High','Low','Close','Volume','MA10','MA50','Return']]
y = df['Target']

# 6. Train-test split (time-series safe)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# 7. Scaling (important improvement)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 8. Model
model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05)
model.fit(X_train, y_train)

# 9. Evaluation
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)
print("Sample Predictions:", y_pred[:5])

# 10. Save model + scaler
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model & Scaler saved")